In [3]:
import pandas as pd
import numpy as np
import warnings
from evcouplings.compare import DistanceMap
warnings.filterwarnings("ignore")

## Prepare the master table of SNPs and indels

In [4]:
#genome_position_to_gene = pd.read_csv("../data/input/genome_position_to_gene_with_codons.csv")

In [5]:
#cp ../../../databases/INDELs\ homoplasy\ count\ across\ lineages\ with\ spacer\ 4.csv ../data/input/

In [6]:
# Read in indel matrix from Roger
# pandas uses xlrd to read excel files, but xlrd removed support for xlsx files, 
# because of course they don't need the most useful file type...

# Input file from Vargas et al 2023
indels = pd.read_csv("../data/input/INDELs homoplasy count across lineages with spacer 4.csv")
print("total number of indels", len(indels))

# Restrict to genes only
indels = indels.query("Gene_Coord != 'None'")
print("Number of indels in a gene", len(indels))

indels = indels.query("INDEL_type != 'frameshift'")
print("Number of non-frameshift indels", len(indels))

indels.rename({"ins/del":"ins_del"}, axis=1, inplace=True)
deletions = indels.query("INDEL_type=='inframe' and ins_del=='del'")
insertions = indels.query("INDEL_type=='inframe' and ins_del=='ins'")

print("Number of inframe deletions", len(deletions))
print("Number of inframe insertions", len(insertions))

grouped_deletions = deletions.groupby(["H37Rv_Locus_Tag", "Codon_Pos", "Position"]).sum()
grouped_deletions = pd.DataFrame(grouped_deletions).reset_index()
grouped_deletions = grouped_deletions.rename({"total":"num_inframe_deletions",
                                                "Codon_Pos":"codon",
                                                "Position":"i",
                                                "H37Rv_Locus_Tag":"locus"
                                               }, axis=1)
grouped_deletions = grouped_deletions[["i", "codon", "locus", "num_inframe_deletions"]]

grouped_insertions = insertions.groupby(["H37Rv_Locus_Tag", "Codon_Pos","Position"]).sum()
grouped_insertions = pd.DataFrame(grouped_insertions).reset_index()
grouped_insertions = grouped_insertions.rename({"total":"num_inframe_insertions",
                                                "Codon_Pos":"codon",
                                                "Position":"i",
                                                "H37Rv_Locus_Tag":"locus"
                                               }, axis=1)
grouped_insertions = grouped_insertions[["i", "codon", "locus", "num_inframe_insertions"]]




total number of indels 46306
Number of indels in a gene 46306
Number of non-frameshift indels 6462
Number of inframe deletions 4685
Number of inframe insertions 1777


In [7]:
## Merge to create the master indel table!
inframe_insertion_count = grouped_insertions
inframe_deletion_count = grouped_deletions

master_indel_count = inframe_insertion_count.merge(inframe_deletion_count, on=['i', 'locus', 'codon'], how="outer")

master_indel_count.fillna(0, inplace=True)
mutations = np.array([master_indel_count.num_inframe_deletions, master_indel_count.num_inframe_insertions]).T

master_indel_count["total_indels"] = np.sum(mutations, axis=1)
master_indel_count = master_indel_count.query("total_indels > 0")
master_indel_count.sort_values("total_indels", ascending=False)

master_indel_count.to_csv("../data/summary_data_proofs/master_indel_count_table.csv")

## Combine indels and mutations into one table

In [8]:
#!cp /n/data1/hms/dbmi/farhat/anna/structure_annotation/homoplasy_count_across_lineages.pkl ../data/input/
snps = pd.read_pickle("../data/input/homoplasy_count_across_lineages.pkl")

# ## Filter out intergenic and synonymous SNPs

snps = snps.query("gene_id != 'intergenic' and AA_change!='None' and SNP_type=='N'")
print(len(snps))
snps["residue"] = [int("".join(list(x)[1:-1])) for x in snps.AA_change]
snps["summed"] = snps["total"]
# ## Merge with indels

indels = pd.read_csv("../data/summary_data_proofs/master_indel_count_table.csv", index_col=0)

indels = indels.query("total_indels > 0")
indels = indels.rename({
    "i": "pos", 
    "locus": "gene_id",
    "total_indels": "summed",
    "codon":"residue"
}, axis=1)
indels["SNP_type"] = "indel"

snps.to_csv("../data/summary_data_proofs/02.SNPs_in_analysis.csv")
indels.to_csv("../data/summary_data_proofs/02.Indels_in_analysis.csv")

snps.pos = snps.pos.astype(int)
all_mutations = pd.concat([snps, indels])
all_mutations.columns

all_mutations = all_mutations[[
    "pos","ref", "alt", "gene_id", "SNP_type", "summed", "residue", "num_inframe_insertions", "num_inframe_deletions"
]]
all_mutations.to_csv("../data/summary_data_proofs/02.all_mutations_in_analysis_prefiltering.csv")



477607


In [9]:
details = pd.concat([snps, indels])
cas10 = details.query("gene_id=='Rv2823c'")
#details.head()



In [10]:
print(len(all_mutations)), print(len(snps)), print(len(indels))

483117
477607
5510


(None, None, None)

#### Filter out the EBR and mappability regions
#### also the blindspots analysis

In [11]:
# Ensure that we've accurately filtered positions from Marin et al
ebr = pd.read_csv(
    "../data/input/EBR.csv", index_col=0
)
# Accurately filter the positions from Modlin et al
modlin = pd.read_csv("../data/input/S7-seq-attributes.csv")[["rv_position", "blindspot"]]

# Filter proteins with low overall mapabbility and EBR
mappability = pd.read_csv("../data/input/AF7_H37Rv_FeatureLevelAnalysis.EBR_And_Pmap.tsv", sep="\t")
mappability = mappability.query('Feature == "CDS"')

proteins_to_keep = mappability.query("Mean_EBR_36CI >= .9 and Mean_PileupMapScore_K50_E4 >= .9")

# Filter out the positions
all_mutations = pd.read_csv("../data/summary_data_proofs/02.all_mutations_in_analysis_prefiltering.csv", index_col=0)
print(len(all_mutations))
all_mutations = all_mutations.merge(ebr, how="left", left_on="pos", right_on="position")
all_mutations = all_mutations.merge(modlin, how="left", left_on="pos", right_on="rv_position")
all_mutations = all_mutations.query("EBR >=0.9 and blindspot==0 and gene_id in @proteins_to_keep.H37rv_GeneID")


483117


In [12]:
all_mutations.to_csv("../data/summary_data_proofs//02.all_mutations_in_analysis_filtered.csv")
len(all_mutations), len(all_mutations.query("SNP_type=='indel'")), len(all_mutations.query("SNP_type=='N'"))

(475046, 5104, 469942)

## Filter for just non-synonymous changes

In [13]:
def mutation_table_cleanup(all_mutations, locus_id, nonsynonymous_only=True):
    
    # prepare the SNP input table
    mutation_count_table = all_mutations.query('gene_id==@locus_id')
    
    # eliminate non-synonymouse mutations, if chosen
    if nonsynonymous_only:
        mutation_count_table= mutation_count_table.query("SNP_type!='S'")
    
    #Sum the number of mutations per columns
    mutation_count_table['summed'] = mutation_count_table.summed.astype(float)
    mutation_count_table = mutation_count_table.groupby("residue").sum()
    mutation_count_table.index.rename("", inplace=True)
    mutation_count_table["residue"] = mutation_count_table.index
    mutation_count_table['residue'] = mutation_count_table.residue.astype(int)
    
    #information about individual positions is irrelevant after summation

    mutation_count_table = mutation_count_table[[
        'residue', 'summed'
    ]]
    
    return mutation_count_table


In [14]:
all_hits = pd.read_csv("../data/summary_data_proofs/01.structure_hits_filtered.csv", index_col=0)
proteins_excluded = []

analyzed_mutations = []

# Determine if there are enough NS mutations to perform this analysis
for idx, row in all_hits.iterrows():
    
    uid = row.uid
    
    to_analyzer = mutation_table_cleanup(all_mutations, row.rv, nonsynonymous_only=True)
    
    if len(to_analyzer) < 2:
        print("too few mutations", uid)
        proteins_excluded.append(uid)
        continue

    # read in the distance map to ensure that all residues analyzed have a structure known
    dm = DistanceMap.from_file(f"../data/filtered_distmaps/{row.Entry}")
    dm.residues_i['id'] = dm.residues_i.id.astype(int)
    to_analyze = to_analyzer.query('residue in @dm.residues_i.id')

    to_analyze.to_csv(f"../data/output/{row.uid}/all_mutations_analyzed.csv")
    to_analyze["uid"] = row.uid
    to_analyze["rv"] = row.rv
    to_analyze["Entry"] = row.Entry

    analyzed_mutations.append(to_analyze)


too few mutations Y2645_MYCTU
too few mutations Y3428_MYCTU
too few mutations VPC16_MYCTU
too few mutations I6X8E6_MYCTU
too few mutations I6Y560_MYCTU
too few mutations O86339_MYCTU
too few mutations O50414_MYCTU
too few mutations P95131_MYCTU
too few mutations Q10844_MYCTU
too few mutations I6Y2Q7_MYCTU
too few mutations I6YAC9_MYCTU
too few mutations I6X9N4_MYCTU
too few mutations O06797_MYCTU
too few mutations I6X043_MYCTU
too few mutations I6XEK2_MYCTU
too few mutations I6Y1F0_MYCTU
too few mutations I6X630_MYCTU
too few mutations I6Y8X8_MYCTU
too few mutations O06370_MYCTU
too few mutations P71949_MYCTU
too few mutations P96366_MYCTU
too few mutations Q79FL0_MYCTU
too few mutations O53402_MYCTU
too few mutations O50413_MYCTU
too few mutations O53337_MYCTU
too few mutations Q10843_MYCTU
too few mutations I6YC39_MYCTU
too few mutations P71641_MYCTU
too few mutations P71639_MYCTU
too few mutations V5QQS8_MYCTU
too few mutations V5QQR7_MYCTU
too few mutations I6X5T4_MYCTU
too few mut

In [15]:
print(len(proteins_excluded), "proteins have too few mutations")

81 proteins have too few mutations


In [21]:
all_hits = all_hits.query("not uid in @proteins_excluded")
all_hits.to_csv("../data/summary_data_proofs/02.structure_hits_filtered_muts.csv", index=False)

In [22]:
all_hits

,uid,Entry,rv,set,pdb_id,pdb_chain,filtered_length
0,Y876_MYCTU,P9WKR5,Rv0876c,setA,NaN,NaN,315.0
2,Y037_MYCTU,P9WJY1,Rv0037c,setA,NaN,NaN,402.0
3,Y1995_MYCTU,P9WLP3,Rv1995,setA,NaN,NaN,189.0
4,Y2012_MYCTU,P9WLM3,Rv2012,setA,NaN,NaN,136.0
5,Y2084_MYCTU,P9WLK1,Rv2084,setA,NaN,NaN,248.0
...,...,...,...,...,...,...,...
3753,O05878_MYCTU,O05878,Rv3233c,setA,NaN,NaN,165.0
3754,I6Y9H2_MYCTU,I6Y9H2,Rv2499c,setA,NaN,NaN,158.0
3755,L0TB61_MYCTU,L0TB61,Rv2037c,setA,NaN,NaN,271.0
3756,L7N662_MYCTU,L7N662,Rv1623c,setA,NaN,NaN,455.0


In [18]:
analyzed_mutations_df = pd.concat(analyzed_mutations)
analyzed_mutations_df.to_csv("../data/summary_data_proofs/02.all_mutations_analyzed.csv")

In [19]:
len(analyzed_mutations_df)

334903

In [20]:
analyzed_mutations_df.query("uid=='CAS10_MYCTU'").sort_values("summed")

,residue,summed,uid,rv,Entry
,,,,,
403.0,403,1.0,CAS10_MYCTU,Rv2823c,P71629
483.0,483,1.0,CAS10_MYCTU,Rv2823c,P71629
485.0,485,1.0,CAS10_MYCTU,Rv2823c,P71629
486.0,486,1.0,CAS10_MYCTU,Rv2823c,P71629
494.0,494,1.0,CAS10_MYCTU,Rv2823c,P71629
...,...,...,...,...,...
310.0,310,3.0,CAS10_MYCTU,Rv2823c,P71629
616.0,616,4.0,CAS10_MYCTU,Rv2823c,P71629
501.0,501,4.0,CAS10_MYCTU,Rv2823c,P71629
